# 4. Imaging: weights, dirty image and PSF

An interferometer samples the Fourier transform of the sky at the baseline
coordinates $(u, v)$, measured in wavelengths. Gridding the visibilities and
inverse Fourier transforming gives the **dirty image**: the true sky convolved
with the **PSF**, the image of a point source through this sampling. Rang's core
reproduces WSClean's Stokes I reader, imaging weights and w-stacking gridder
(research/core-i-imaging-replay.md).

With four antennas the uv coverage is sparse, so the PSF has strong sidelobes
and the dirty image is dominated by them. That is exactly why the campaigns
measure the **point-source residual**: the dirty image minus the peak times the
PSF.

In [ ]:
%matplotlib inline
import os

os.environ.setdefault("RAYON_NUM_THREADS", "4")  # share the cores; set before the first core call

import numpy as np
import pandas as pd

from rang.notebook import data, experiment, flags, gains, image, steer, survey

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

# Downloads and the extracted fixture go here; choose your own with RANG_NOTEBOOK_CACHE.
CACHE = data.default_cache_dir()
# LUNO sections run when a local copy of the slice exists (RANG_NOTEBOOK_LUNO=0 turns them off).
LUNO = os.environ.get("RANG_NOTEBOOK_LUNO", "1") != "0" and data.find_luno_slice() is not None
print("cache:", CACHE, "| LUNO slice available:", LUNO)

In [ ]:
ds = data.load_fixture(cache_dir=CACHE)
pd.Series(data.summary(ds), name="value").to_frame()

In [ ]:
image.uv_coverage(ds)

**What to look for:** six baselines trace short arcs during the scan (Earth
rotation), mirrored through the origin because $V(-u,-v) = V^*(u,v)$; the
4096 channels spread each arc radially.

## Corrected data

Images are made from corrected data $G_p^{-1} V_{pq} G_q^{-H}$, with the gains of a
solve (notebook 3). Stokes I is $(XX + YY)/2$.

In [ ]:
sol = gains.solve(ds)
corrected = image.image(ds, sol)  # Briggs weighting, robust 0.5, 512 x 512 pixels of 8 arcsec
image.image_figure(corrected)

The peak should equal the model flux at the phase centre (the table's
`flux_recovery` is peak divided by the mean model Stokes I). The raw,
uncorrected data, imaged the same way, show why calibration matters:

In [ ]:
raw = image.image(ds, None)
display(image.metrics_table({"corrected": corrected, "raw DATA": raw}))
image.compare(corrected, raw, ("corrected", "raw"))

## Weighting

Each visibility gets an imaging weight. **Natural** weighting uses the noise
weights: best sensitivity, broad PSF. **Uniform** weighting divides by the local
density of samples in the uv plane: sharper PSF, more noise. **Briggs**
weighting interpolates with one parameter, the *robustness* $R$: $R=-2$ is close
to uniform, $R=+2$ close to natural.

In [ ]:
natural = image.image(ds, sol, mode="natural")
uniform = image.image(ds, sol, mode="uniform")
display(image.weights_figure(ds, natural))
image.weights_figure(ds, uniform)

In [ ]:
sweep = image.robust_sweep(ds, sol, robusts=(-1.0, 0.5, 2.0))
sweep

**What to look for:** as robustness rises the PSF's far sidelobes and the
residual change, the peak barely moves. On four stations no weighting removes
the sidelobes; that needs more baselines (or deconvolution, which this project
does not use for its decisions).

## Comparing two flag sets

A flagging method changes which cells are imaged. Here the original flags are
compared with the teaching strategy's additions (notebook 2), each solved and
imaged with the same settings. The difference image shows where the
removed cells contributed.

In [ ]:
teaching = flags.run_strategy(ds, flags.strategy_library()["teaching"])
flag_sets = {"original": np.zeros_like(ds.state["train"]), "teaching strategy": teaching["additions_on_train"]}
panel = steer.ImagePanel(ds, flag_sets)
result = panel.update("original", "teaching strategy", "briggs", 0.5, True)
display(result["table"])
result["compare"]

### What a professional checks

- the PSF centre is 1 and its peak sits at the image centre;
- the weight sums match between methods that should match (they are in the
  metrics: `weights: visibility_weight_sum`);
- a lower residual RMS after flagging can simply mean *fewer data*: compare
  imaged cells and off-source RMS too.

## Steer it yourself

In [ ]:
panel

## Optional: the image in CARTA

[CARTA](https://cartavis.org) shows large images and cubes with tiled
rendering, profiles and moments. `rang.notebook.carta` writes the dirty image
to FITS and opens it in a running CARTA backend (Linux or macOS; CARTA has no
Windows build). Set `RANG_NOTEBOOK_CARTA=1` and start a backend first
(research/nb-carta.md); otherwise the cell prints its skip line.

In [ ]:
from IPython.display import Image
from rang.notebook import carta

if not carta.switch():
    print(carta.skip_line())
elif not carta.available():
    print(carta.skip_line(carta.problem()))
else:
    header = carta.image_header(ds, corrected)
    session = carta.open(corrected, wcs=header, name="04-corrected-dirty.fits")
    png = carta.snapshot(session, carta.session_dir() / "04-corrected-dirty.png")
    carta.close(session)  # a session in your own browser stays open
    display(Image(filename=str(png)))